In [ ]:
# ============================================================
# CELDA 0.1 — Importaciones globales
# ============================================================

import os
import math
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, DoubleType, FloatType, NumericType

print("✅ Librerías importadas correctamente")


## ⚙️ SECCIÓN 0 — Configuración del entorno


# Cambia este valor por el nombre real de tu bucket
BUCKET_NAME = "hcdr"  # Ejemplo: "hcdr-proyecto-eafit-2026"

RAW_PATH     = "s3://{}/raw/".format(BUCKET_NAME)
TRUSTED_PATH = "s3://{}/trusted/".format(BUCKET_NAME)
REFINED_PATH = "s3://{}/refined/eda/".format(BUCKET_NAME)

# Bases de datos esperadas en Glue Catalog
RAW_DB = "raw_db"
TRUSTED_DB = "hcdr_trusted"

# Si ya tienes tablas catalogadas en Glue, deja esto en True.
# Si estás probando local/temporalmente desde CSV, ponlo en False.
USE_GLUE_CATALOG = True

# Si alguna tabla catalogada no existe, el notebook puede usar S3 como fallback.
ALLOW_S3_FALLBACK = False

# Fracción para graficar variables numéricas. Evita hacer toPandas() de toda la tabla.
SAMPLE_FRACTION = 0.03

PATHS = {
    "application_train"      : RAW_PATH + "application_train/application_train.csv",
    "bureau"                 : RAW_PATH + "bureau/bureau.csv",
    "bureau_balance"         : RAW_PATH + "bureau_balance/bureau_balance.csv",
    "previous_application"   : RAW_PATH + "previous_application/previous_application.csv",
    "pos_cash_balance"       : RAW_PATH + "pos_cash_balance/POS_CASH_balance.csv",
    "credit_card_balance"    : RAW_PATH + "credit_card_balance/credit_card_balance.csv",
    "installments_payments"  : RAW_PATH + "installments_payments/installments_payments.csv",
}

print("✅ Configuración lista")
print("RAW_PATH     :", RAW_PATH)
print("TRUSTED_PATH :", TRUSTED_PATH)
print("REFINED_PATH :", REFINED_PATH)
print("RAW_DB       :", RAW_DB)
print("USE_GLUE_CATALOG:", USE_GLUE_CATALOG)



In [ ]:
# ============================================================
# CELDA 0.1 — Importaciones globales
# ============================================================

import os
import math
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, DoubleType, FloatType, NumericType

print("✅ Librerías importadas correctamente")


## ⚙️ SECCIÓN 0 — Configuración del entorno


# Cambia este valor por el nombre real de tu bucket
BUCKET_NAME = "hcdr"  # Ejemplo: "hcdr-proyecto-eafit-2026"

RAW_PATH     = "s3://{}/raw/".format(BUCKET_NAME)
TRUSTED_PATH = "s3://{}/trusted/".format(BUCKET_NAME)
REFINED_PATH = "s3://{}/refined/eda/".format(BUCKET_NAME)

# Bases de datos esperadas en Glue Catalog
RAW_DB = "raw_db"
TRUSTED_DB = "hcdr_trusted"

# Si ya tienes tablas catalogadas en Glue, deja esto en True.
# Si estás probando local/temporalmente desde CSV, ponlo en False.
USE_GLUE_CATALOG = True

# Si alguna tabla catalogada no existe, el notebook puede usar S3 como fallback.
ALLOW_S3_FALLBACK = False

# Fracción para graficar variables numéricas. Evita hacer toPandas() de toda la tabla.
SAMPLE_FRACTION = 0.03

PATHS = {
    "application_train"      : RAW_PATH + "application_train/application_train.csv",
    "bureau"                 : RAW_PATH + "bureau/bureau.csv",
    "bureau_balance"         : RAW_PATH + "bureau_balance/bureau_balance.csv",
    "previous_application"   : RAW_PATH + "previous_application/previous_application.csv",
    "pos_cash_balance"       : RAW_PATH + "pos_cash_balance/POS_CASH_balance.csv",
    "credit_card_balance"    : RAW_PATH + "credit_card_balance/credit_card_balance.csv",
    "installments_payments"  : RAW_PATH + "installments_payments/installments_payments.csv",
}

print("✅ Configuración lista")
print("RAW_PATH     :", RAW_PATH)
print("TRUSTED_PATH :", TRUSTED_PATH)
print("REFINED_PATH :", REFINED_PATH)
print("RAW_DB       :", RAW_DB)
print("USE_GLUE_CATALOG:", USE_GLUE_CATALOG)



# ============================================================
# CELDA 0.3 — Inicialización de Spark / Glue
# ============================================================

try:
    from awsglue.context import GlueContext
    from pyspark.context import SparkContext

    sc = SparkContext.getOrCreate()
    glueContext = GlueContext(sc)
    spark = glueContext.spark_session
    print("✅ Spark inicializado desde AWS GlueContext")
except Exception as e:
    print("⚠️ No se pudo usar GlueContext. Se usará SparkSession estándar.")
    spark = (
        SparkSession.builder
        .appName("EDA_HomeCreditRisk_SparkSQL_AWS")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.shuffle.partitions", "120")
        .enableHiveSupport()
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

# ============================================================
# CELDA 0.3 — Inicialización de Spark / Glue
# ============================================================

try:
    from awsglue.context import GlueContext
    from pyspark.context import SparkContext

    sc = SparkContext.getOrCreate()
    glueContext = GlueContext(sc)
    spark = glueContext.spark_session
    print("✅ Spark inicializado desde AWS GlueContext")
except Exception as e:
    print("⚠️ No se pudo usar GlueContext. Se usará SparkSession estándar.")
    spark = (
        SparkSession.builder
        .appName("EDA_HomeCreditRisk_SparkSQL_AWS")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.shuffle.partitions", "120")
        .enableHiveSupport()
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


# ============================================================
# CELDA 0.4 — Funciones auxiliares para SQL, tablas y gráficos
# ============================================================

def normalize_column_names(df):
    """Convierte nombres de columnas a minúsculas para evitar problemas entre CSV, Glue y Athena."""
    for old in df.columns:
        new = old.strip().lower().replace(" ", "_")
        if old != new:
            df = df.withColumnRenamed(old, new)
    return df


def load_table(view_name, catalog_table=None, s3_path=None):
    """Carga una tabla desde Glue Catalog si es posible; si falla, usa CSV desde S3 como fallback."""
    catalog_table = catalog_table or view_name

    if USE_GLUE_CATALOG:
        try:
            full_name = "{}.{}".format(RAW_DB, catalog_table)
            print("📚 Leyendo tabla catalogada:", full_name)
            df = spark.table(full_name)
            df = normalize_column_names(df)
            df.createOrReplaceTempView(view_name)
            print("   ✅ Vista SparkSQL creada: {} | {:,} filas × {} columnas".format(view_name, df.count(), len(df.columns)))
            return df
        except Exception as e:
            print("⚠️ No se pudo leer desde Glue Catalog:", catalog_table)
            print("   Motivo:", str(e)[:250])
            if not ALLOW_S3_FALLBACK:
                raise e

    if s3_path is None:
        raise ValueError("No hay ruta S3 para fallback de {}".format(view_name))

    print("📥 Leyendo CSV desde S3:", s3_path)
    df = spark.read.csv(s3_path, header=True, inferSchema=True, nullValue="", nanValue="NaN")
    df = normalize_column_names(df)
    df.createOrReplaceTempView(view_name)
    print("   ✅ Vista SparkSQL creada: {} | {:,} filas × {} columnas".format(view_name, df.count(), len(df.columns)))
    return df


def sql(query, rows=20, truncate=False):
    """Ejecuta SparkSQL y muestra resultados."""
    result = spark.sql(query)
    result.show(rows, truncate=truncate)
    return result


def guardar_spark_en_refined(df, nombre):
    """Guarda un Spark DataFrame en refined/silver como CSV con header."""
    out_path = REFINED_PATH + nombre.strip("/") + "/"
    (
        df.coalesce(1)
          .write
          .mode("overwrite")
          .option("header", "true")
          .csv(out_path)
    )
    print("💾 Guardado en refined/silver:", out_path)


def guardar_parquet_trusted(df, nombre):
    """Guarda un Spark DataFrame en trusted/gold como Parquet."""
    out_path = TRUSTED_PATH + nombre.strip("/") + "/"
    (
        df.write
          .mode("overwrite")
          .option("compression", "snappy")
          .parquet(out_path)
    )
    print("💾 Guardado en trusted/gold:", out_path)


def plot_bar_from_pandas(pdf, x, y, title, xlabel=None, ylabel=None, rotation=0):
    if pdf.empty:
        print("⚠️ DataFrame vacío. No se genera gráfico.")
        return
    plt.figure(figsize=(10, 4))
    plt.bar(pdf[x].astype(str), pdf[y])
    plt.title(title)
    plt.xlabel(xlabel or x)
    plt.ylabel(ylabel or y)
    plt.xticks(rotation=rotation, ha="right" if rotation else "center")
    plt.tight_layout()
    plt.show()


def plot_hbar_from_pandas(pdf, x, y, title, xlabel=None, ylabel=None):
    if pdf.empty:
        print("⚠️ DataFrame vacío. No se genera gráfico.")
        return
    plt.figure(figsize=(10, max(4, len(pdf) * 0.35)))
    plt.barh(pdf[y].astype(str), pdf[x])
    plt.title(title)
    plt.xlabel(xlabel or x)
    plt.ylabel(ylabel or y)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()


def build_null_sql(view_name, columns):
    """Construye SQL para reporte vertical de nulos por columna."""
    selects = []
    stack_entries = []
    for i, c in enumerate(columns):
        alias = "n_{}".format(i)
        selects.append("SUM(CASE WHEN `{}` IS NULL THEN 1 ELSE 0 END) AS {}".format(c, alias))
        stack_entries.append("'{}', {}".format(c, alias))

    query = """
    WITH base AS (
        SELECT
            COUNT(*) AS total,
            {selects}
        FROM {view_name}
    )
    SELECT
        columna,
        nulos,
        ROUND(100.0 * nulos / total, 2) AS pct_nulos
    FROM base
    LATERAL VIEW stack({n}, {stack_entries}) s AS columna, nulos
    ORDER BY pct_nulos DESC, nulos DESC
    """.format(
        selects=",\n            ".join(selects),
        view_name=view_name,
        n=len(columns),
        stack_entries=", ".join(stack_entries)
    )
    return query


def value_counts_sql(view_name, column_name, target_col="target"):
    """Distribución y tasa de default para una columna categórica."""
    return """
    SELECT
        '{col}' AS variable,
        CAST(`{col}` AS STRING) AS valor,
        COUNT(*) AS total,
        SUM(CASE WHEN `{target}` = 1 THEN 1 ELSE 0 END) AS defaults,
        ROUND(100.0 * SUM(CASE WHEN `{target}` = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS default_rate_pct
    FROM {view}
    WHERE `{col}` IS NOT NULL
      AND `{target}` IN (0, 1)
    GROUP BY `{col}`
    ORDER BY total DESC
    """.format(col=column_name, target=target_col, view=view_name)

print("✅ Funciones auxiliares definidas")

# ============================================================
# CELDA 1.1 — Carga de todas las tablas como vistas SparkSQL
# ============================================================

TABLES = {}

for name, path in PATHS.items():
    TABLES[name] = load_table(name, catalog_table=name, s3_path=path)

# DataFrames principales
app = TABLES["application_train"]
bureau = TABLES["bureau"]
bureau_balance = TABLES["bureau_balance"]
previous_application = TABLES["previous_application"]
pos_cash_balance = TABLES["pos_cash_balance"]
credit_card_balance = TABLES["credit_card_balance"]
installments_payments = TABLES["installments_payments"]

print("\n✅ Todas las tablas quedaron disponibles como vistas SparkSQL")
spark.sql("SHOW TABLES").show(truncate=False)

# ============================================================
# CELDA 1.2 — Inventario general del datalake raw con SparkSQL
# ============================================================

inventory_rows = []

for nombre, df in TABLES.items():
    temp_view = "tmp_inv_{}".format(nombre)
    spark.sql("""
        SELECT
            '{nombre}' AS tabla,
            COUNT(*) AS filas
        FROM {nombre}
    """.format(nombre=nombre)).createOrReplaceTempView(temp_view)

inventory_query = " UNION ALL ".join([
    "SELECT * FROM tmp_inv_{}".format(nombre) for nombre in TABLES.keys()
])

inventario_sql = spark.sql(inventory_query)
inventario_sql.show(truncate=False)
guardar_spark_en_refined(inventario_sql, "inventario_raw")

# ============================================================
# CELDA 1.3 — Vista de columnas disponibles por tabla
# ============================================================

schema_rows = []
for nombre, df in TABLES.items():
    for field in df.schema.fields:
        schema_rows.append((nombre, field.name, str(field.dataType)))

schema_df = spark.createDataFrame(schema_rows, ["tabla", "columna", "tipo"])
schema_df.createOrReplaceTempView("schema_inventario")

sql("""
SELECT tabla, columna, tipo
FROM schema_inventario
ORDER BY tabla, columna
""", rows=200, truncate=False)

guardar_spark_en_refined(schema_df, "schema_inventario")






In [ ]:
# Ejecutar análisis de nulos solo para la tabla bureau
print("🔍 Analizando nulos en tabla: bureau")

query_nulos = build_null_sql_referenciado("bureau", TABLES["bureau"].columns)

null_analysis_by_table = spark.sql(query_nulos)

# Crear vista SparkSQL global
null_analysis_by_table.createOrReplaceTempView("null_analysis_by_table")

print("✅ Vista creada: null_analysis_by_table")

sql("""
SELECT
    tabla,
    columna,
    total_filas,
    nulos,
    pct_nulos
FROM null_analysis_by_table
ORDER BY pct_nulos DESC, tabla, columna
""", rows=300, truncate=False)

In [ ]:
# ============================================================
# Consulta — DAYS_ENDDATE_FACT vs CREDIT_ACTIVE
# Lectura de negocio de nulos en bureau
# ============================================================

days_enddate_fact_vs_credit_active = sql("""
SELECT
    credit_active AS estado_credito,
    COUNT(*) AS total_creditos,
    SUM(CASE WHEN days_enddate_fact IS NULL THEN 1 ELSE 0 END) AS nulos_days_enddate_fact,
    ROUND(
        100.0 * SUM(CASE WHEN days_enddate_fact IS NULL THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS pct_nulo_days_enddate_fact
FROM bureau
GROUP BY credit_active
ORDER BY pct_nulo_days_enddate_fact DESC
""", rows=100, truncate=False)

In [ ]:
# ============================================================
# Consulta — Nulos de amt_credit_max_overdue según credit_day_overdue
# ============================================================

amt_credit_max_overdue_nulls_vs_day_overdue = sql("""
WITH nulos_base AS (
    SELECT
        *
    FROM bureau
    WHERE amt_credit_max_overdue IS NULL
),

clasificacion AS (
    SELECT
        CASE
            WHEN credit_day_overdue > 0 THEN 'credit_day_overdue > 0'
            WHEN credit_day_overdue = 0 THEN 'credit_day_overdue = 0'
            WHEN credit_day_overdue IS NULL THEN 'credit_day_overdue nulo'
            ELSE 'otro caso'
        END AS grupo_credit_day_overdue,
        COUNT(*) AS total_nulos_amt_credit_max_overdue
    FROM nulos_base
    GROUP BY
        CASE
            WHEN credit_day_overdue > 0 THEN 'credit_day_overdue > 0'
            WHEN credit_day_overdue = 0 THEN 'credit_day_overdue = 0'
            WHEN credit_day_overdue IS NULL THEN 'credit_day_overdue nulo'
            ELSE 'otro caso'
        END
),

total AS (
    SELECT
        COUNT(*) AS total_nulos
    FROM nulos_base
)

SELECT
    'bureau' AS tabla,
    'amt_credit_max_overdue' AS columna_nula_analizada,
    c.grupo_credit_day_overdue,
    c.total_nulos_amt_credit_max_overdue,
    t.total_nulos,
    ROUND(
        100.0 * c.total_nulos_amt_credit_max_overdue / t.total_nulos,
        2
    ) AS pct_sobre_nulos_amt_credit_max_overdue
FROM clasificacion c
CROSS JOIN total t
ORDER BY pct_sobre_nulos_amt_credit_max_overdue DESC
""", rows=100, truncate=False)

In [ ]:
# ============================================================
# Consulta — Relación entre nulo de max overdue, días de mora y monto vencido
# Tabla: bureau
# ============================================================

max_overdue_mora_actual = sql("""
WITH clasificacion AS (
    SELECT
        CASE
            WHEN amt_credit_max_overdue IS NULL THEN 'max_overdue_nulo'
            ELSE 'max_overdue_no_nulo'
        END AS grupo_max_overdue,

        CASE
            WHEN credit_day_overdue > 0 THEN 'tiene_dias_mora_actual'
            WHEN credit_day_overdue = 0 THEN 'sin_dias_mora_actual'
            ELSE 'credit_day_overdue_nulo'
        END AS grupo_dias_mora,

        CASE
            WHEN amt_credit_sum_overdue > 0 THEN 'tiene_monto_vencido_actual'
            WHEN amt_credit_sum_overdue = 0 THEN 'sin_monto_vencido_actual'
            ELSE 'amt_credit_sum_overdue_nulo'
        END AS grupo_monto_vencido
    FROM bureau
),

conteo AS (
    SELECT
        grupo_max_overdue,
        grupo_dias_mora,
        grupo_monto_vencido,
        COUNT(*) AS total_registros
    FROM clasificacion
    GROUP BY
        grupo_max_overdue,
        grupo_dias_mora,
        grupo_monto_vencido
),

total AS (
    SELECT COUNT(*) AS total_bureau
    FROM bureau
)

SELECT
    c.grupo_max_overdue,
    c.grupo_dias_mora,
    c.grupo_monto_vencido,
    c.total_registros,
    ROUND(
        100.0 * c.total_registros / t.total_bureau,
        2
    ) AS pct_total_bureau
FROM conteo c
CROSS JOIN total t
ORDER BY c.total_registros DESC
""", rows=100, truncate=False)

In [ ]:
# ============================================================
# Distribución por rangos — amt_credit_max_overdue
# ============================================================

rangos_amt_credit_max_overdue = sql("""
SELECT
    CASE
        WHEN amt_credit_max_overdue IS NULL THEN 'nulo'
        WHEN amt_credit_max_overdue = 0 THEN '0'
        WHEN amt_credit_max_overdue > 0 AND amt_credit_max_overdue <= 1000 THEN '0 - 1.000'
        WHEN amt_credit_max_overdue > 1000 AND amt_credit_max_overdue <= 10000 THEN '1.000 - 10.000'
        WHEN amt_credit_max_overdue > 10000 AND amt_credit_max_overdue <= 100000 THEN '10.000 - 100.000'
        WHEN amt_credit_max_overdue > 100000 AND amt_credit_max_overdue <= 1000000 THEN '100.000 - 1.000.000'
        WHEN amt_credit_max_overdue > 1000000 THEN 'mayor a 1.000.000'
        ELSE 'otro'
    END AS rango_amt_credit_max_overdue,

    COUNT(*) AS total_registros,

    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM bureau),
        2
    ) AS pct_total_bureau

FROM bureau
GROUP BY
    CASE
        WHEN amt_credit_max_overdue IS NULL THEN 'nulo'
        WHEN amt_credit_max_overdue = 0 THEN '0'
        WHEN amt_credit_max_overdue > 0 AND amt_credit_max_overdue <= 1000 THEN '0 - 1.000'
        WHEN amt_credit_max_overdue > 1000 AND amt_credit_max_overdue <= 10000 THEN '1.000 - 10.000'
        WHEN amt_credit_max_overdue > 10000 AND amt_credit_max_overdue <= 100000 THEN '10.000 - 100.000'
        WHEN amt_credit_max_overdue > 100000 AND amt_credit_max_overdue <= 1000000 THEN '100.000 - 1.000.000'
        WHEN amt_credit_max_overdue > 1000000 THEN 'mayor a 1.000.000'
        ELSE 'otro'
    END
ORDER BY total_registros DESC
""", rows=100, truncate=False)

In [ ]:
# ============================================================
# Análisis univariado — amt_credit_max_overdue
# Tabla: bureau
# ============================================================

analisis_amt_credit_max_overdue = sql("""
WITH base AS (
    SELECT
        CAST(amt_credit_max_overdue AS DOUBLE) AS amt_credit_max_overdue
    FROM bureau
),

valores_no_nulos AS (
    SELECT
        amt_credit_max_overdue
    FROM base
    WHERE amt_credit_max_overdue IS NOT NULL
),

estadisticas AS (
    SELECT
        COUNT(*) AS total_no_nulos,

        SUM(CASE WHEN amt_credit_max_overdue = 0 THEN 1 ELSE 0 END) AS valores_cero,
        SUM(CASE WHEN amt_credit_max_overdue > 0 THEN 1 ELSE 0 END) AS valores_mayores_cero,

        MIN(amt_credit_max_overdue) AS minimo,
        percentile_approx(amt_credit_max_overdue, 0.25) AS q1,
        percentile_approx(amt_credit_max_overdue, 0.50) AS mediana,
        percentile_approx(amt_credit_max_overdue, 0.75) AS q3,
        AVG(amt_credit_max_overdue) AS promedio,
        percentile_approx(amt_credit_max_overdue, 0.90) AS p90,
        percentile_approx(amt_credit_max_overdue, 0.95) AS p95,
        percentile_approx(amt_credit_max_overdue, 0.99) AS p99,
        MAX(amt_credit_max_overdue) AS maximo
    FROM valores_no_nulos
),

limites AS (
    SELECT
        total_no_nulos,
        valores_cero,
        valores_mayores_cero,

        minimo,
        q1,
        mediana,
        q3,
        promedio,
        p90,
        p95,
        p99,
        maximo,

        q3 - q1 AS iqr,
        q1 - 1.5 * (q3 - q1) AS limite_inferior_iqr,
        q3 + 1.5 * (q3 - q1) AS limite_superior_iqr
    FROM estadisticas
),

outliers AS (
    SELECT
        COUNT(*) AS outliers_superiores
    FROM valores_no_nulos v
    CROSS JOIN limites l
    WHERE v.amt_credit_max_overdue > l.limite_superior_iqr
)

SELECT
    'bureau' AS tabla,
    'amt_credit_max_overdue' AS columna,

    l.total_no_nulos,

    l.valores_cero,
    ROUND(100.0 * l.valores_cero / l.total_no_nulos, 2) AS pct_valores_cero_sobre_no_nulos,

    l.valores_mayores_cero,
    ROUND(100.0 * l.valores_mayores_cero / l.total_no_nulos, 2) AS pct_valores_mayores_cero_sobre_no_nulos,

    ROUND(l.minimo, 2) AS minimo,
    ROUND(l.q1, 2) AS q1_percentil_25,
    ROUND(l.mediana, 2) AS mediana_percentil_50,
    ROUND(l.q3, 2) AS q3_percentil_75,
    ROUND(l.promedio, 2) AS promedio,
    ROUND(l.p90, 2) AS percentil_90,
    ROUND(l.p95, 2) AS percentil_95,
    ROUND(l.p99, 2) AS percentil_99,
    ROUND(l.maximo, 2) AS maximo,

    ROUND(l.iqr, 2) AS iqr,
    ROUND(l.limite_inferior_iqr, 2) AS limite_inferior_iqr,
    ROUND(l.limite_superior_iqr, 2) AS limite_superior_iqr,

    o.outliers_superiores,
    ROUND(100.0 * o.outliers_superiores / l.total_no_nulos, 2) AS pct_outliers_superiores_sobre_no_nulos

FROM limites l
CROSS JOIN outliers o
""", rows=20, truncate=False)

In [ ]:
# ============================================================
# Correlación entre cuota anual y monto total del crédito
# Tabla: bureau
# Columnas: amt_annuity vs amt_credit_sum
# ============================================================

correlacion_annuity_credit_sum = sql("""
WITH base AS (
    SELECT
        CAST(amt_annuity AS DOUBLE) AS amt_annuity,
        CAST(amt_credit_sum AS DOUBLE) AS amt_credit_sum
    FROM bureau
),

totales AS (
    SELECT
        COUNT(*) AS total_registros,
        SUM(CASE WHEN amt_annuity IS NULL THEN 1 ELSE 0 END) AS nulos_amt_annuity,
        SUM(CASE WHEN amt_credit_sum IS NULL THEN 1 ELSE 0 END) AS nulos_amt_credit_sum,
        SUM(CASE WHEN amt_annuity IS NOT NULL AND amt_credit_sum IS NOT NULL THEN 1 ELSE 0 END) AS pares_validos
    FROM base
),

validos AS (
    SELECT
        amt_annuity,
        amt_credit_sum
    FROM base
    WHERE amt_annuity IS NOT NULL
      AND amt_credit_sum IS NOT NULL
)

SELECT
    'bureau' AS tabla,
    'amt_annuity' AS variable_1,
    'amt_credit_sum' AS variable_2,

    t.total_registros,
    t.nulos_amt_annuity,
    t.nulos_amt_credit_sum,
    t.pares_validos,

    ROUND(100.0 * t.pares_validos / t.total_registros, 2) AS pct_pares_validos,

    ROUND(CORR(v.amt_annuity, v.amt_credit_sum), 4) AS correlacion_pearson,

    ROUND(AVG(v.amt_annuity), 2) AS promedio_amt_annuity,
    ROUND(AVG(v.amt_credit_sum), 2) AS promedio_amt_credit_sum,

    ROUND(STDDEV(v.amt_annuity), 2) AS desviacion_amt_annuity,
    ROUND(STDDEV(v.amt_credit_sum), 2) AS desviacion_amt_credit_sum

FROM validos v
CROSS JOIN totales t
GROUP BY
    t.total_registros,
    t.nulos_amt_annuity,
    t.nulos_amt_credit_sum,
    t.pares_validos
""", rows=20, truncate=False)

In [ ]:
# ============================================================
# Versión limpia: excluye posible fila de encabezado mal cargada
# ============================================================

nulos_annuity_por_credit_type_active_limpio = sql("""
SELECT
    credit_type,
    credit_active,
    COUNT(*) AS total_registros,

    SUM(CASE WHEN amt_annuity IS NULL THEN 1 ELSE 0 END) AS total_nulos_annuity,

    SUM(CASE WHEN amt_annuity IS NOT NULL THEN 1 ELSE 0 END) AS total_no_nulos_annuity,

    ROUND(
        100.0 * SUM(CASE WHEN amt_annuity IS NULL THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS pct_nulos_annuity,

    ROUND(
        100.0 * SUM(CASE WHEN amt_annuity IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS pct_no_nulos_annuity

FROM bureau
WHERE credit_type <> 'CREDIT_TYPE'

GROUP BY
    credit_type,
    credit_active

ORDER BY
    pct_nulos_annuity DESC,
    total_registros DESC
""", rows=300, truncate=False)

In [ ]:
# ============================================================
# Relación entre disponibilidad de amt_annuity en bureau
# y tasa de default en application_train
# ============================================================

annuity_cliente_default = sql("""
WITH annuity_cliente AS (
    SELECT
        CAST(sk_id_curr AS BIGINT) AS sk_id_curr,
        COUNT(*) AS total_creditos_bureau,
        SUM(CASE WHEN amt_annuity IS NOT NULL THEN 1 ELSE 0 END) AS creditos_con_annuity,
        SUM(CASE WHEN amt_annuity IS NULL THEN 1 ELSE 0 END) AS creditos_sin_annuity
    FROM bureau
    WHERE CAST(sk_id_curr AS STRING) <> 'SK_ID_CURR'
    GROUP BY CAST(sk_id_curr AS BIGINT)
),

base_clientes AS (
    SELECT
        CAST(a.sk_id_curr AS BIGINT) AS sk_id_curr,
        CAST(a.target AS DOUBLE) AS target,
        b.total_creditos_bureau,
        b.creditos_con_annuity,
        b.creditos_sin_annuity
    FROM application_train a
    LEFT JOIN annuity_cliente b
        ON CAST(a.sk_id_curr AS BIGINT) = b.sk_id_curr
)

SELECT
    CASE
        WHEN creditos_con_annuity IS NULL THEN 'cliente_sin_bureau'
        WHEN creditos_con_annuity = 0 THEN 'cliente_sin_annuity_reportada'
        WHEN creditos_con_annuity > 0 THEN 'cliente_con_alguna_annuity'
    END AS grupo_annuity,

    COUNT(*) AS total_clientes,

    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_clientes,

    ROUND(
        100.0 * AVG(target),
        2
    ) AS default_rate_pct,

    ROUND(
        AVG(total_creditos_bureau),
        2
    ) AS promedio_creditos_bureau,

    ROUND(
        AVG(creditos_con_annuity),
        2
    ) AS promedio_creditos_con_annuity,

    ROUND(
        AVG(creditos_sin_annuity),
        2
    ) AS promedio_creditos_sin_annuity

FROM base_clientes

GROUP BY
    CASE
        WHEN creditos_con_annuity IS NULL THEN 'cliente_sin_bureau'
        WHEN creditos_con_annuity = 0 THEN 'cliente_sin_annuity_reportada'
        WHEN creditos_con_annuity > 0 THEN 'cliente_con_alguna_annuity'
    END

ORDER BY default_rate_pct DESC
""", rows=100, truncate=False)

In [ ]:
# ============================================================
# Exploración 2 — Nulos de days_enddate_fact por credit_active
# ============================================================

days_enddate_fact_por_estado = sql("""
SELECT
    credit_active,
    COUNT(*) AS total_registros,

    SUM(CASE WHEN days_enddate_fact IS NULL THEN 1 ELSE 0 END) AS total_nulos_days_enddate_fact,

    SUM(CASE WHEN days_enddate_fact IS NOT NULL THEN 1 ELSE 0 END) AS total_no_nulos_days_enddate_fact,

    ROUND(
        100.0 * SUM(CASE WHEN days_enddate_fact IS NULL THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS pct_nulos_days_enddate_fact,

    ROUND(
        100.0 * SUM(CASE WHEN days_enddate_fact IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS pct_no_nulos_days_enddate_fact

FROM bureau
WHERE credit_active <> 'CREDIT_ACTIVE'
GROUP BY credit_active
ORDER BY pct_nulos_days_enddate_fact DESC
""", rows=100, truncate=False)

In [ ]:
# ============================================================
# Validación de negocio:
# ¿La fecha real de cierre del crédito diferencia la tasa de default?
# Variable: days_enddate_fact
# Tabla bureau + application_train
# ============================================================

default_por_recencia_cierre = sql("""
WITH cierre_cliente AS (
    SELECT
        CAST(sk_id_curr AS BIGINT) AS sk_id_curr,

        COUNT(*) AS total_creditos_bureau,

        SUM(CASE WHEN days_enddate_fact IS NOT NULL THEN 1 ELSE 0 END) AS creditos_con_fecha_cierre,

        SUM(CASE WHEN days_enddate_fact IS NULL THEN 1 ELSE 0 END) AS creditos_sin_fecha_cierre,

        -- Como los días son negativos, el valor más cercano a 0 representa el cierre más reciente
        MAX(CASE 
                WHEN days_enddate_fact IS NOT NULL THEN CAST(days_enddate_fact AS INT)
            END) AS cierre_mas_reciente_dias,

        MIN(CASE 
                WHEN days_enddate_fact IS NOT NULL THEN CAST(days_enddate_fact AS INT)
            END) AS cierre_mas_antiguo_dias

    FROM bureau
    WHERE CAST(sk_id_curr AS STRING) <> 'SK_ID_CURR'
    GROUP BY CAST(sk_id_curr AS BIGINT)
),

base_clientes AS (
    SELECT
        CAST(a.sk_id_curr AS BIGINT) AS sk_id_curr,
        CAST(a.target AS INT) AS target,

        c.total_creditos_bureau,
        c.creditos_con_fecha_cierre,
        c.creditos_sin_fecha_cierre,
        c.cierre_mas_reciente_dias,
        c.cierre_mas_antiguo_dias,

        CASE
            WHEN c.sk_id_curr IS NULL THEN '00_cliente_sin_bureau'
            WHEN c.creditos_con_fecha_cierre = 0 THEN '01_sin_fecha_cierre_reportada'
            WHEN c.cierre_mas_reciente_dias > 0 THEN '02_cierre_futuro_revisar'
            WHEN c.cierre_mas_reciente_dias BETWEEN -30 AND 0 THEN '03_cierre_ultimo_mes'
            WHEN c.cierre_mas_reciente_dias BETWEEN -90 AND -31 THEN '04_cierre_1_a_3_meses'
            WHEN c.cierre_mas_reciente_dias BETWEEN -180 AND -91 THEN '05_cierre_3_a_6_meses'
            WHEN c.cierre_mas_reciente_dias BETWEEN -365 AND -181 THEN '06_cierre_6_a_12_meses'
            WHEN c.cierre_mas_reciente_dias BETWEEN -730 AND -366 THEN '07_cierre_1_a_2_anios'
            WHEN c.cierre_mas_reciente_dias BETWEEN -1825 AND -731 THEN '08_cierre_2_a_5_anios'
            WHEN c.cierre_mas_reciente_dias < -1825 THEN '09_cierre_mas_5_anios'
            ELSE '10_otro'
        END AS grupo_recencia_cierre

    FROM application_train a
    LEFT JOIN cierre_cliente c
        ON CAST(a.sk_id_curr AS BIGINT) = c.sk_id_curr
),

default_global AS (
    SELECT
        AVG(target) AS default_rate_global
    FROM base_clientes
)

SELECT
    b.grupo_recencia_cierre,

    COUNT(*) AS total_clientes,

    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_clientes,

    SUM(CASE WHEN b.target = 1 THEN 1 ELSE 0 END) AS total_defaults,

    SUM(CASE WHEN b.target = 0 THEN 1 ELSE 0 END) AS total_no_defaults,

    ROUND(
        100.0 * AVG(b.target),
        2
    ) AS default_rate_pct,

    ROUND(
        100.0 * g.default_rate_global,
        2
    ) AS default_rate_global_pct,

    ROUND(
        100.0 * (AVG(b.target) - g.default_rate_global),
        2
    ) AS diferencia_vs_global_puntos_pct,

    ROUND(
        AVG(b.target) / g.default_rate_global,
        3
    ) AS lift_vs_global,

    ROUND(
        AVG(b.total_creditos_bureau),
        2
    ) AS promedio_creditos_bureau,

    ROUND(
        AVG(b.creditos_con_fecha_cierre),
        2
    ) AS promedio_creditos_con_fecha_cierre

FROM base_clientes b
CROSS JOIN default_global g

GROUP BY
    b.grupo_recencia_cierre,
    g.default_rate_global

ORDER BY
    b.grupo_recencia_cierre
""", rows=100, truncate=False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Total de clientes en application_train
total_clientes = app.count()

# 2. Clientes con inconsistencia:
# crédito activo pero con fecha real de cierre
clientes_con_inconsistencia = (
    bureau
    .join(
        app.select("sk_id_curr", "target"),
        on="sk_id_curr",
        how="left"
    )
    .filter(
        (F.col("credit_active") == "Active") &
        (F.col("days_enddate_fact").isNotNull())
    )
    .select("sk_id_curr", "target")
    .distinct()
)

# 3. Agrupar por target
resultado = (
    clientes_con_inconsistencia
    .withColumn(
        "target_grupo",
        F.coalesce(F.col("target").cast("string"), F.lit("SIN_TARGET"))
    )
    .groupBy("target_grupo")
    .agg(
        F.countDistinct("sk_id_curr").alias("clientes_con_inconsistencia")
    )
)

# 4. Calcular porcentajes
window_total = Window.partitionBy()

resultado = (
    resultado
    .withColumn(
        "pct_sobre_total_clientes",
        F.round(
            F.lit(100.0) * F.col("clientes_con_inconsistencia") / F.lit(total_clientes),
            4
        )
    )
    .withColumn(
        "pct_dentro_inconsistencias",
        F.round(
            F.lit(100.0) * F.col("clientes_con_inconsistencia") /
            F.sum("clientes_con_inconsistencia").over(window_total),
            2
        )
    )
    .orderBy("target_grupo")
)

resultado.show(truncate=False)

In [ ]:
# ============================================================
# CELDA — Estadísticas de DAYS_CREDIT_ENDDATE cuando NO es nulo
# ============================================================

stats_enddate = sql("""
SELECT
    COUNT(*) AS total_no_nulos,

    MIN(days_credit_enddate) AS min_days_credit_enddate,

    percentile_approx(days_credit_enddate, 0.25) AS q1_days_credit_enddate,

    percentile_approx(days_credit_enddate, 0.50) AS mediana_days_credit_enddate,

    percentile_approx(days_credit_enddate, 0.75) AS q3_days_credit_enddate,

    MAX(days_credit_enddate) AS max_days_credit_enddate,

    ROUND(AVG(days_credit_enddate), 2) AS avg_days_credit_enddate

FROM bureau
WHERE days_credit_enddate IS NOT NULL
""", rows=10, truncate=False)

guardar_spark_en_refined(
    stats_enddate,
    "bureau_stats_days_credit_enddate"
)

In [ ]:
rangos_enddate_target = sql("""
SELECT
    COALESCE(CAST(a.target AS STRING), 'SIN_TARGET') AS target,
    CASE
        WHEN b.days_credit_enddate IS NULL THEN 'NULL'
        WHEN b.days_credit_enddate < 0 THEN 'fecha esperada ya vencida'
        WHEN b.days_credit_enddate >= 0 THEN 'fecha esperada futura'
    END AS tipo_enddate,
    COUNT(*) AS total_registros,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY a.target), 2) AS pct_dentro_target
FROM bureau b
LEFT JOIN application_train a
    ON b.sk_id_curr = a.sk_id_curr
GROUP BY
    a.target,
    CASE
        WHEN b.days_credit_enddate IS NULL THEN 'NULL'
        WHEN b.days_credit_enddate < 0 THEN 'fecha esperada ya vencida'
        WHEN b.days_credit_enddate >= 0 THEN 'fecha esperada futura'
    END
ORDER BY target, total_registros DESC
""", rows=50, truncate=False)

guardar_spark_en_refined(rangos_enddate_target, "bureau_rangos_days_credit_enddate_target")